In [1]:
import os
os.makedirs("/kaggle/working/belt-monitor/simulator", exist_ok=True)
open("/kaggle/working/belt-monitor/simulator/__init__.py", "w").close()
%cd /kaggle/working/belt-monitor

/kaggle/working/belt-monitor


In [3]:
%%writefile /kaggle/working/belt-monitor/simulator/config.py
"""Every number the simulator uses, in one place."""

CONVEYOR_LENGTH_M = 3000          # distance from tail pulley to head pulley
BELT_LOOP_M = 6000                # the belt goes out and comes back
BELT_SPEED_MPS = 4.0              # design speed
BELT_WIDTH_MM = 1600

SENSOR_IDS = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"]
SENSOR_POSITION_M = {"S1": 300, "S2": 600, "S3": 900, "S4": 1200, "S5": 1500,
                     "S6": 1800, "S7": 2100, "S8": 2400, "S9": 2700, "S10": 3000}

READING_INTERVAL_S = 60           # each sensor sends one message per minute

CHANNELS = ["temperature_c", "vibration_mm_s", "belt_speed_mps", "drift_mm"]

Writing /kaggle/working/belt-monitor/simulator/config.py


In [4]:
!python -c "from simulator import config; print(config.SENSOR_POSITION_M['S3'])"

900


In [5]:
%%writefile /kaggle/working/belt-monitor/simulator/config.py
"""Every number the simulator uses, in one place.

The backend imports the belt geometry and VALID_RANGE from here too, so the
simulator and the backend can never disagree about them.
"""

# ---------------------------------------------------------------- belt
CONVEYOR_LENGTH_M = 3000          # distance from tail pulley to head pulley
BELT_LOOP_M = 6000                # the belt goes out and comes back
BELT_SPEED_MPS = 4.0              # design speed; one lap = 6000 / 4 = 1500 s = 25 min
BELT_WIDTH_MM = 1600

SENSOR_IDS = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"]
# S1 is 300 m from the tail pulley, S10 is at 3000 m, which is the head (drive) pulley.
SENSOR_POSITION_M = {"S1": 300, "S2": 600, "S3": 900, "S4": 1200, "S5": 1500,
                     "S6": 1800, "S7": 2100, "S8": 2400, "S9": 2700, "S10": 3000}

READING_INTERVAL_S = 60           # each sensor sends one message per minute

CHANNELS = ["temperature_c", "vibration_mm_s", "belt_speed_mps", "drift_mm"]

# ---------------------------------------------------------------- healthy signals
# Air temperature follows a daily wave: 36 C at 15:00, 20 C at 03:00.
AMBIENT_MEAN_C = 28.0
AMBIENT_SWING_C = 8.0
AMBIENT_HOTTEST_HOUR = 15

BEARING_RISE_C = 22.0             # a healthy roller bearing runs this much above the air
VIBRATION_BASE_MM_S = 2.0         # healthy vibration velocity (RMS)

# Random noise added to every reading (standard deviation).
NOISE_SD = {"temperature_c": 0.8, "vibration_mm_s": 0.25,
            "belt_speed_mps": 0.02, "drift_mm": 6.0}

# Each sensor point gets one fixed offset per channel, drawn once from
# -OFFSET_MAX..+OFFSET_MAX. No two sensor points read exactly the same.
OFFSET_MAX = {"temperature_c": 3.0, "vibration_mm_s": 0.3,
              "belt_speed_mps": 0.0, "drift_mm": 5.0}

# ---------------------------------------------------------------- failures
# A failure episode has three moments:
#   onset    -> the fault starts to develop
#   failure  -> the fault is fully developed (the values below are reached)
#   repaired -> maintenance has fixed it; readings go back to healthy
# Real faults take days. The durations are compressed to hours so that a
# 60-day run holds enough episodes to train on. The README must say so.
FAILURE_DURATION_H = {"seized_roller": (8, 24),   # (shortest, longest) onset -> failure
                      "belt_drift": (6, 18),
                      "belt_slip": (4, 12)}
FAILURE_MEAN_GAP_H = 36           # average quiet time between two episodes of one type
REPAIR_MINUTES = (30, 90)         # failure -> repaired

ROLLER_VIBRATION_ADD = 7.0        # mm/s added at failure. Grows as progress ** 1.5 (early)
ROLLER_TEMPERATURE_ADD = 50.0     # deg C added at failure. Grows as progress ** 3 (late)
DRIFT_ADD_MM = 150.0              # sideways shift at failure. Grows as progress ** 2
SLIP_SPEED_LOSS = 0.20            # share of speed lost at failure (4.0 -> 3.2 m/s)

# ---------------------------------------------------------------- messy data
P_DROPPED = 0.005                 # chance a message is never sent
P_NULL_FIELD = 0.003              # chance one value inside a message is null
P_SPIKE = 0.002                   # chance one value is replaced by a wild number
P_LATE = 0.01                     # chance a message arrives late
LATE_MINUTES = (2, 15)

OUTAGE_MEAN_GAP_H = 240           # per sensor: one silent period about every 10 days
OUTAGE_MINUTES = (10, 120)
STUCK_MEAN_GAP_H = 240            # per sensor: one frozen channel about every 10 days
STUCK_MINUTES = (30, 180)

# A value outside these limits cannot be a real measurement. Half of all
# spikes are pushed outside them, the other half stay inside.
VALID_RANGE = {"temperature_c": (-20.0, 200.0),
               "vibration_mm_s": (0.0, 50.0),
               "belt_speed_mps": (0.0, 8.0),
               "drift_mm": (-800.0, 800.0)}      # 800 mm = half the belt width

IMPOSSIBLE_SPIKES = {"temperature_c": [-273.0, 999.0],
                     "vibration_mm_s": [-5.0, 500.0],
                     "belt_speed_mps": [-3.0, 40.0],
                     "drift_mm": [-5000.0, 5000.0]}
# A spike that stays inside the valid range: the true value plus one of these.
PLAUSIBLE_SPIKE_ADD = {"temperature_c": (25.0, 60.0),
                       "vibration_mm_s": (5.0, 20.0),
                       "belt_speed_mps": (-2.5, -1.0),
                       "drift_mm": (100.0, 300.0)}

DECIMALS = {"temperature_c": 2, "vibration_mm_s": 3, "belt_speed_mps": 3, "drift_mm": 1}

Overwriting /kaggle/working/belt-monitor/simulator/config.py


In [6]:
%%writefile /kaggle/working/belt-monitor/simulator/generate.py
"""Conveyor belt sensor simulator.

One BeltSimulator object produces three things:
  ticks()      -> the sensor messages, minute by minute, in the order they ARRIVE
  failures     -> ground truth: every failure episode that was injected
  data_faults  -> ground truth: every piece of messy data that was injected

Time is counted in whole minutes from the start. "m" always means "minute number".
"""
import math
import random
from datetime import datetime, timedelta

from simulator import config as C


class BeltSimulator:
    def __init__(self, days, seed, start="2026-09-01T00:00:00Z", messy=True):
        self.rng = random.Random(seed)          # same seed -> exactly the same data
        self.total_min = int(days * 24 * 60)
        self.start = datetime.fromisoformat(start.replace("Z", "+00:00"))
        self.messy = messy                      # False -> no messy data, failures only
        self.data_faults = []

        # One fixed offset per sensor per channel.
        self.offsets = {}
        for sid in C.SENSOR_IDS:
            self.offsets[sid] = {}
            for ch in C.CHANNELS:
                self.offsets[sid][ch] = self.rng.uniform(-C.OFFSET_MAX[ch], C.OFFSET_MAX[ch])

        # Everything that has a start and an end is decided now, before any reading exists.
        self.failures = self._schedule_failures()
        self.outages = self._schedule_windows("outage", C.OUTAGE_MEAN_GAP_H, C.OUTAGE_MINUTES)
        self.stuck = self._schedule_windows("stuck", C.STUCK_MEAN_GAP_H, C.STUCK_MINUTES)

    # ------------------------------------------------------------ helpers
    def ts(self, m):
        """Minute number -> timestamp text, e.g. 2026-09-01T00:05:00Z."""
        return (self.start + timedelta(minutes=m)).strftime("%Y-%m-%dT%H:%M:%SZ")

    def ambient(self, m):
        """Air temperature at minute m: a daily wave, hottest at 15:00."""
        t = self.start + timedelta(minutes=m)
        hour = t.hour + t.minute / 60
        angle = 2 * math.pi * (hour - C.AMBIENT_HOTTEST_HOUR) / 24
        return C.AMBIENT_MEAN_C + C.AMBIENT_SWING_C * math.cos(angle)

    def _log_fault(self, fault_type, sensor_id, channel, start_m, end_m, detail=""):
        self.data_faults.append({"fault_type": fault_type, "sensor_id": sensor_id,
                                 "channel": channel, "start_ts": self.ts(start_m),
                                 "end_ts": self.ts(end_m), "detail": detail})

    # ------------------------------------------------------------ schedules
    def _schedule_failures(self):
        failures = []
        for ftype in C.FAILURE_DURATION_H:
            shortest_h, longest_h = C.FAILURE_DURATION_H[ftype]
            m = int(self.rng.expovariate(1 / (C.FAILURE_MEAN_GAP_H * 60)))
            while True:
                duration = self.rng.randint(shortest_h * 60, longest_h * 60)
                repair = self.rng.randint(C.REPAIR_MINUTES[0], C.REPAIR_MINUTES[1])
                if m + duration + repair >= self.total_min:
                    break
                # Slip happens at the drive pulley and slows the whole belt,
                # so it belongs to no single sensor.
                sensor_id = "ALL" if ftype == "belt_slip" else self.rng.choice(C.SENSOR_IDS)
                new = {"type": ftype, "sensor_id": sensor_id, "onset_min": m,
                       "failure_min": m + duration, "repaired_min": m + duration + repair,
                       "direction": self.rng.choice([-1, 1])}
                # Keep it only if this sensor has no other episode at the same time.
                clash = False
                for old in failures:
                    if (old["sensor_id"] == sensor_id and new["onset_min"] < old["repaired_min"]
                            and old["onset_min"] < new["repaired_min"]):
                        clash = True
                if not clash:
                    failures.append(new)
                gap = int(self.rng.expovariate(1 / (C.FAILURE_MEAN_GAP_H * 60)))
                m = m + duration + repair + gap

        failures.sort(key=lambda f: f["onset_min"])
        for number, f in enumerate(failures, start=1):
            f["failure_id"] = f"F{number:03d}"
            f["onset_ts"] = self.ts(f["onset_min"])
            f["failure_ts"] = self.ts(f["failure_min"])
            f["repaired_ts"] = self.ts(f["repaired_min"])
        return failures

    def _schedule_windows(self, kind, mean_gap_h, minutes_range):
        """Outage and stuck periods. Returns {sensor_id: [(start_m, end_m, channel), ...]}."""
        windows = {sid: [] for sid in C.SENSOR_IDS}
        if not self.messy:
            return windows
        for sid in C.SENSOR_IDS:
            m = int(self.rng.expovariate(1 / (mean_gap_h * 60)))
            while m < self.total_min:
                length = self.rng.randint(minutes_range[0], minutes_range[1])
                end_m = min(m + length, self.total_min - 1)
                channel = "all" if kind == "outage" else self.rng.choice(C.CHANNELS)
                windows[sid].append((m, end_m, channel))
                self._log_fault(kind, sid, channel, m, end_m)
                m = end_m + int(self.rng.expovariate(1 / (mean_gap_h * 60)))
        return windows

    @staticmethod
    def _window_at(windows, m):
        for w in windows:
            if w[0] <= m <= w[1]:
                return w
        return None

    # ------------------------------------------------------------ signal model
    def _healthy(self, sid, ambient):
        off = self.offsets[sid]
        g = self.rng.gauss
        return {
            "temperature_c": ambient + C.BEARING_RISE_C + off["temperature_c"]
                             + g(0, C.NOISE_SD["temperature_c"]),
            "vibration_mm_s": max(0.0, C.VIBRATION_BASE_MM_S + off["vibration_mm_s"]
                                  + g(0, C.NOISE_SD["vibration_mm_s"])),
            "belt_speed_mps": C.BELT_SPEED_MPS + g(0, C.NOISE_SD["belt_speed_mps"]),
            "drift_mm": off["drift_mm"] + g(0, C.NOISE_SD["drift_mm"]),
        }

    @staticmethod
    def progress(f, m):
        """0.0 at onset, 1.0 at failure, stays 1.0 until repaired."""
        p = (m - f["onset_min"]) / (f["failure_min"] - f["onset_min"])
        return min(1.0, p)

    def _apply_failure(self, values, f, m):
        p = self.progress(f, m)
        if f["type"] == "seized_roller":
            values["vibration_mm_s"] += C.ROLLER_VIBRATION_ADD * p ** 1.5
            values["temperature_c"] += C.ROLLER_TEMPERATURE_ADD * p ** 3
        elif f["type"] == "belt_drift":
            values["drift_mm"] += f["direction"] * C.DRIFT_ADD_MM * p ** 2
        elif f["type"] == "belt_slip":
            values["belt_speed_mps"] *= 1 - C.SLIP_SPEED_LOSS * p ** 2

    # ------------------------------------------------------------ main loop
    def ticks(self):
        """Yield (arrival_ts, [messages]) once per minute.

        A message that is late is held back and comes out in a later minute,
        so its own "ts" is older than the arrival_ts it comes out with.
        """
        rng = self.rng
        pending = {}        # arrival minute -> messages that will arrive then
        frozen = {}         # (sensor, stuck window start) -> the value it is stuck at

        for m in range(self.total_min):
            ambient = self.ambient(m)
            active = [f for f in self.failures if f["onset_min"] <= m < f["repaired_min"]]

            for sid in C.SENSOR_IDS:
                # 1. The true physical values.
                values = self._healthy(sid, ambient)
                for f in active:
                    if f["sensor_id"] == sid or f["sensor_id"] == "ALL":
                        self._apply_failure(values, f, m)
                for ch in C.CHANNELS:
                    values[ch] = round(values[ch], C.DECIMALS[ch])

                delay = 0
                if self.messy:
                    # 2. Outage: the sensor is silent, no message at all.
                    if self._window_at(self.outages[sid], m):
                        continue
                    # 3. Dropped: this single message is never sent.
                    if rng.random() < C.P_DROPPED:
                        self._log_fault("dropped", sid, "all", m, m)
                        continue
                    # 4. Stuck: one channel repeats the value it had when it froze.
                    w = self._window_at(self.stuck[sid], m)
                    if w:
                        key = (sid, w[0])
                        if key not in frozen:
                            frozen[key] = values[w[2]]
                        values[w[2]] = frozen[key]
                    # 5. Spike: one value replaced by a wild number.
                    if rng.random() < C.P_SPIKE:
                        ch = rng.choice(C.CHANNELS)
                        if rng.random() < 0.5:
                            values[ch] = rng.choice(C.IMPOSSIBLE_SPIKES[ch])
                            kind = "impossible"
                        else:
                            low, high = C.PLAUSIBLE_SPIKE_ADD[ch]
                            values[ch] = round(values[ch] + rng.uniform(low, high), C.DECIMALS[ch])
                            kind = "plausible"
                        self._log_fault("spike", sid, ch, m, m, f"{kind} value={values[ch]}")
                    # 6. Null field: the message arrives but one value is empty.
                    if rng.random() < C.P_NULL_FIELD:
                        ch = rng.choice(C.CHANNELS)
                        values[ch] = None
                        self._log_fault("null_field", sid, ch, m, m)
                    # 7. Late: the message is sent, but arrives some minutes later.
                    if rng.random() < C.P_LATE:
                        delay = rng.randint(C.LATE_MINUTES[0], C.LATE_MINUTES[1])
                        self._log_fault("late", sid, "all", m, m, f"delay_min={delay}")

                message = {"sensor_id": sid, "ts": self.ts(m)}
                message.update(values)
                pending.setdefault(m + delay, []).append(message)

            yield self.ts(m), pending.pop(m, [])

        # Messages that were still on their way when the run ended.
        for m in sorted(pending):
            yield self.ts(m), pending[m]

Writing /kaggle/working/belt-monitor/simulator/generate.py


In [7]:
!cd /kaggle/working/belt-monitor && python -c "from simulator.generate import BeltSimulator; sim = BeltSimulator(1, 42); arrival, messages = next(sim.ticks()); print(arrival); print(messages[0]); print(len(messages), 'messages in the first minute')"

2026-09-01T00:00:00Z
{'sensor_id': 'S1', 'ts': '2026-09-01T00:00:00Z', 'temperature_c': 44.73, 'vibration_mm_s': 1.507, 'belt_speed_mps': 4.019, 'drift_mm': -6.2}
10 messages in the first minute


In [8]:
%%writefile /kaggle/working/belt-monitor/simulator/run.py
"""Command line for the simulator.

History for training (writes three CSV files):
    python -m simulator.run batch --days 60 --seed 42 --out data/train

Live feed for the backend (one HTTP POST per simulated minute):
    python -m simulator.run live --url http://127.0.0.1:8000/readings --seed 7 --speedup 600
"""
import argparse
import csv
import json
import os
import time
import urllib.error
import urllib.request

from simulator import config as C
from simulator.generate import BeltSimulator

FAILURE_COLUMNS = ["failure_id", "type", "sensor_id", "onset_ts", "failure_ts", "repaired_ts"]
FAULT_COLUMNS = ["fault_type", "sensor_id", "channel", "start_ts", "end_ts", "detail"]


def write_csv(path, columns, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def run_batch(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    count = 0
    with open(os.path.join(args.out, "readings.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["arrival_ts", "sensor_id", "ts"] + C.CHANNELS)
        for arrival_ts, messages in sim.ticks():
            for msg in messages:
                # A null value is written as an empty cell.
                row = [arrival_ts, msg["sensor_id"], msg["ts"]]
                for ch in C.CHANNELS:
                    row.append("" if msg[ch] is None else msg[ch])
                writer.writerow(row)
                count += 1
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)

    expected = sim.total_min * len(C.SENSOR_IDS)
    print(f"readings written : {count} of {expected} possible")
    for ftype in C.FAILURE_DURATION_H:
        print(f"{ftype:17}: {sum(1 for x in sim.failures if x['type'] == ftype)} episodes")
    for kind in ["outage", "stuck", "spike", "null_field", "dropped", "late"]:
        print(f"{kind:17}: {sum(1 for x in sim.data_faults if x['fault_type'] == kind)}")


def post_json(url, payload):
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=body,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=5) as response:
        return response.status


def run_live(args):
    sim = BeltSimulator(args.days, args.seed, args.start, messy=not args.clean)
    os.makedirs(args.out, exist_ok=True)
    # The failure schedule is known before the first reading, so write it now.
    write_csv(os.path.join(args.out, "failures.csv"), FAILURE_COLUMNS, sim.failures)
    pause = C.READING_INTERVAL_S / args.speedup      # real seconds per simulated minute
    sent = failed = 0
    try:
        for arrival_ts, messages in sim.ticks():
            if messages:
                try:
                    post_json(args.url, messages)
                    sent += len(messages)
                except (urllib.error.URLError, OSError) as error:
                    failed += len(messages)
                    print(f"{arrival_ts} POST failed: {error}")
            time.sleep(pause)
    except KeyboardInterrupt:
        print("stopped by user")
    finally:
        write_csv(os.path.join(args.out, "data_faults.csv"), FAULT_COLUMNS, sim.data_faults)
        print(f"messages sent: {sent}, not delivered: {failed}")


def main():
    parser = argparse.ArgumentParser(description="Conveyor belt sensor simulator")
    sub = parser.add_subparsers(dest="mode", required=True)
    for name in ["batch", "live"]:
        p = sub.add_parser(name)
        p.add_argument("--days", type=float, default=60)
        p.add_argument("--seed", type=int, default=42)
        p.add_argument("--start", default="2026-09-01T00:00:00Z")
        p.add_argument("--clean", action="store_true", help="no messy data, failures only")
        p.add_argument("--out", default="data/train" if name == "batch" else "data/live")
        if name == "live":
            p.add_argument("--url", default="http://127.0.0.1:8000/readings")
            p.add_argument("--speedup", type=float, default=600,
                           help="600 means one simulated minute takes 0.1 real seconds")
    args = parser.parse_args()
    if args.mode == "batch":
        run_batch(args)
    else:
        run_live(args)


if __name__ == "__main__":
    main()

Writing /kaggle/working/belt-monitor/simulator/run.py


In [9]:
!cd /kaggle/working/belt-monitor && python -m simulator.run batch --days 60 --seed 42 --out data/train
!head -4 /kaggle/working/belt-monitor/data/train/failures.csv

readings written : 856001 of 864000 possible
seized_roller    : 27 episodes
belt_drift       : 29 episodes
belt_slip        : 30 episodes
outage           : 53
stuck            : 60
spike            : 1751
null_field       : 2516
dropped          : 4323
late             : 8478
failure_id,type,sensor_id,onset_ts,failure_ts,repaired_ts
F001,belt_slip,ALL,2026-09-02T04:12:00Z,2026-09-02T10:46:00Z,2026-09-02T11:55:00Z
F002,belt_drift,S6,2026-09-02T18:08:00Z,2026-09-03T05:27:00Z,2026-09-03T06:22:00Z
F003,seized_roller,S4,2026-09-02T19:53:00Z,2026-09-03T04:39:00Z,2026-09-03T05:51:00Z


In [10]:
import os
os.makedirs("/kaggle/working/belt-monitor/backend", exist_ok=True)
open("/kaggle/working/belt-monitor/backend/__init__.py", "w").close()

In [11]:
%%writefile /kaggle/working/belt-monitor/backend/db.py
"""The database: one SQLite file with three tables."""
import os
import sqlite3

# The file sits in belt-monitor/data/belt.db unless BELT_DB says otherwise.
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DB_PATH = os.environ.get("BELT_DB") or os.path.join(PROJECT_DIR, "data", "belt.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS readings (
    sensor_id       TEXT NOT NULL,
    ts              TEXT NOT NULL,      -- when the sensor measured
    temperature_c   REAL,               -- NULL means missing or impossible
    vibration_mm_s  REAL,
    belt_speed_mps  REAL,
    drift_mm        REAL,
    flags           TEXT NOT NULL,      -- what was wrong with this reading, '' if nothing
    PRIMARY KEY (sensor_id, ts)         -- the same reading cannot be stored twice
);
CREATE INDEX IF NOT EXISTS readings_by_ts ON readings (ts);

CREATE TABLE IF NOT EXISTS rejected (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    reason  TEXT NOT NULL,
    raw     TEXT NOT NULL               -- the message exactly as it came in
);

CREATE TABLE IF NOT EXISTS anomalies (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id   TEXT NOT NULL,          -- 'ALL' for belt slip
    position_m  INTEGER NOT NULL,
    type        TEXT NOT NULL,
    severity    TEXT NOT NULL,          -- 'warning' or 'critical'
    value       REAL,                   -- the latest value that broke the limit
    started_ts  TEXT NOT NULL,
    last_ts     TEXT NOT NULL,
    ended_ts    TEXT                    -- NULL while the anomaly is still active
);
"""


def connect(path=None):
    path = path or DB_PATH
    os.makedirs(os.path.dirname(path), exist_ok=True)
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row          # rows can be read by column name
    conn.execute("PRAGMA journal_mode=WAL")  # the dashboard can read while the backend writes
    conn.executescript(SCHEMA)
    return conn

Writing /kaggle/working/belt-monitor/backend/db.py


In [12]:
!cd /kaggle/working/belt-monitor && python -c "from backend.db import connect; conn = connect(); print([r['name'] for r in conn.execute('SELECT name FROM sqlite_master WHERE type = \"table\" ORDER BY name')])"

['anomalies', 'readings', 'rejected', 'sqlite_sequence']


In [13]:
%%writefile /kaggle/working/belt-monitor/backend/validate.py
"""Check one incoming message. Nothing is ever guessed or repaired here."""
from datetime import datetime

from simulator import config as C

TS_FORMAT = "%Y-%m-%dT%H:%M:%SZ"


def validate(message):
    """Return (row, flags, reason).

    reason is None  -> the message is usable. row holds the cleaned values,
                       flags lists what was wrong with single values.
    reason is text  -> the whole message is rejected, row is None.
    """
    if not isinstance(message, dict):
        return None, [], "not a JSON object"
    if message.get("sensor_id") not in C.SENSOR_IDS:
        return None, [], "unknown sensor_id"
    try:
        datetime.strptime(message.get("ts"), TS_FORMAT)
    except (TypeError, ValueError):
        return None, [], "bad timestamp"

    row = {"sensor_id": message["sensor_id"], "ts": message["ts"]}
    flags = []
    for ch in C.CHANNELS:
        value = message.get(ch)
        low, high = C.VALID_RANGE[ch]
        if value is None:
            row[ch] = None
            flags.append("missing:" + ch)
        elif isinstance(value, bool) or not isinstance(value, (int, float)) or value != value:
            row[ch] = None                       # text, true/false or NaN
            flags.append("not_a_number:" + ch)
        elif value < low or value > high:
            row[ch] = None                       # cannot be a real measurement
            flags.append("out_of_range:" + ch)
        else:
            row[ch] = float(value)
    return row, flags, None

Writing /kaggle/working/belt-monitor/backend/validate.py


In [14]:
%%writefile /kaggle/working/belt-monitor/backend/analyse.py
"""Turn stored readings into anomalies.

A limit must be broken by 3 readings in a row before an anomaly opens, and
3 readings in a row must be back inside the limit before it closes.
One wild reading therefore changes nothing.
"""
from datetime import datetime
from statistics import median

from simulator import config as C
from backend.validate import TS_FORMAT

IN_A_ROW = 3
STUCK_AFTER = 5              # the same value this many times in a row = stuck sensor
OFFLINE_AFTER_MIN = 5        # no reading for this many minutes = sensor offline

# anomaly type: (channel, warning limit, critical limit)
LIMITS = {
    "high_temperature": ("temperature_c", 70.0, 85.0),
    "high_vibration": ("vibration_mm_s", 4.5, 7.1),
    "belt_drift": ("drift_mm", 75.0, 120.0),          # distance from centre, either side
}
SLIP_WARNING = 0.95 * C.BELT_SPEED_MPS                 # 3.8 m/s
SLIP_CRITICAL = 0.85 * C.BELT_SPEED_MPS                # 3.4 m/s
DRIVE_PULLEY_M = C.CONVEYOR_LENGTH_M


def level(values, warning, critical, low_is_bad=False):
    """Judge the newest 3 values. Returns 'critical', 'warning', 'normal' or 'unknown'."""
    if len(values) < IN_A_ROW or None in values:
        return "unknown"                                # not enough trusted readings
    if low_is_bad:
        values = [-v for v in values]
        warning, critical = -warning, -critical
    if all(v > critical for v in values):
        return "critical"
    if all(v > warning for v in values):
        return "warning"
    if all(v <= warning for v in values):
        return "normal"
    return "unknown"                                    # mixed: keep the current state


def set_anomaly(conn, active, sensor_id, position_m, kind, verdict, ts, value):
    """Open, update or close the anomaly (sensor_id, kind) according to the verdict."""
    key = (sensor_id, kind)
    if verdict == "unknown":
        return
    if verdict == "normal":
        if key in active:
            conn.execute("UPDATE anomalies SET ended_ts = ? WHERE id = ?", (ts, active[key]))
            del active[key]
        return
    if key in active:
        conn.execute("UPDATE anomalies SET severity = ?, value = ?, last_ts = ? WHERE id = ?",
                     (verdict, value, ts, active[key]))
    else:
        cur = conn.execute(
            "INSERT INTO anomalies (sensor_id, position_m, type, severity, value, started_ts, last_ts)"
            " VALUES (?, ?, ?, ?, ?, ?, ?)", (sensor_id, position_m, kind, verdict, value, ts, ts))
        active[key] = cur.lastrowid


def analyse(conn, sensor_ids):
    """Run every rule. sensor_ids = the sensors that just sent a reading."""
    active = {(r["sensor_id"], r["type"]): r["id"] for r in
              conn.execute("SELECT id, sensor_id, type FROM anomalies WHERE ended_ts IS NULL")}
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    if clock is None:
        return

    for sid in C.SENSOR_IDS:
        position = C.SENSOR_POSITION_M[sid]
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                              (sid, STUCK_AFTER)).fetchall()
        if not newest:
            continue

        # Rule 1: sensor offline. Its newest reading is too far behind the clock.
        behind = datetime.strptime(clock, TS_FORMAT) - datetime.strptime(newest[0]["ts"], TS_FORMAT)
        minutes = behind.total_seconds() / 60
        verdict = "warning" if minutes > OFFLINE_AFTER_MIN else "normal"
        set_anomaly(conn, active, sid, position, "sensor_offline", verdict, clock, minutes)
        if sid not in sensor_ids:
            continue                                    # nothing new from this sensor

        # Rule 2: stuck sensor. One channel shows the exact same value 5 times in a row.
        stuck = []
        for ch in C.CHANNELS:
            values = [r[ch] for r in newest]
            if len(values) == STUCK_AFTER and values[0] is not None and len(set(values)) == 1:
                stuck.append(ch)
        if stuck:
            flags = newest[0]["flags"].split(",") if newest[0]["flags"] else []
            flags += ["stuck:" + ch for ch in stuck if "stuck:" + ch not in flags]
            conn.execute("UPDATE readings SET flags = ? WHERE sensor_id = ? AND ts = ?",
                         (",".join(flags), sid, newest[0]["ts"]))
        set_anomaly(conn, active, sid, position, "sensor_stuck",
                    "warning" if stuck else "normal", newest[0]["ts"], None)

        # Rule 3: limits on temperature, vibration and drift.
        for kind, (ch, warning, critical) in LIMITS.items():
            values = []
            for r in newest[:IN_A_ROW]:
                trusted = r[ch] is not None and "stuck:" + ch not in r["flags"] and ch not in stuck
                values.append(abs(r[ch]) if trusted else None)
            set_anomaly(conn, active, sid, position, kind, level(values, warning, critical),
                        newest[0]["ts"], values[0])

    # Rule 4: belt slip. The belt is one piece, so its speed is judged once per minute
    # as the middle value (median) of all sensors, not sensor by sensor.
    rows = conn.execute("SELECT ts, belt_speed_mps FROM readings WHERE ts IN "
                        "(SELECT DISTINCT ts FROM readings ORDER BY ts DESC LIMIT ?)",
                        (IN_A_ROW,)).fetchall()
    by_minute = {}
    for r in rows:
        by_minute.setdefault(r["ts"], [])
        if r["belt_speed_mps"] is not None:
            by_minute[r["ts"]].append(r["belt_speed_mps"])
    speeds = []
    for ts in sorted(by_minute, reverse=True):
        speeds.append(median(by_minute[ts]) if len(by_minute[ts]) >= 5 else None)
    set_anomaly(conn, active, "ALL", DRIVE_PULLEY_M, "belt_slip",
                level(speeds, SLIP_WARNING, SLIP_CRITICAL, low_is_bad=True), clock,
                speeds[0] if speeds else None)

Writing /kaggle/working/belt-monitor/backend/analyse.py


In [15]:
!cd /kaggle/working/belt-monitor && python -c "from backend.analyse import level; print(level([72, 71, 73], 70, 85)); print(level([90, 88, 86], 70, 85)); print(level([72, None, 73], 70, 85)); print(level([60, 61, 59], 70, 85)); print(level([72, 60, 73], 70, 85)); print(level([3.7, 3.75, 3.79], 3.8, 3.4, low_is_bad=True))"

warning
critical
unknown
normal
unknown
warning


In [16]:
%%writefile /kaggle/working/belt-monitor/backend/ingest.py
"""Receive one batch of messages: validate, store, analyse."""
import json

from backend.analyse import analyse
from backend.validate import validate


def ingest(conn, messages, commit=True):
    """Store a list of messages. Returns counts of what happened to them."""
    summary = {"received": len(messages), "stored": 0, "rejected": 0, "duplicate": 0, "late": 0}
    # The backend's clock is the newest timestamp it has stored so far.
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    touched = set()

    for message in messages:
        row, flags, reason = validate(message)
        if reason is not None:
            conn.execute("INSERT INTO rejected (reason, raw) VALUES (?, ?)",
                         (reason, json.dumps(message, default=str)))
            summary["rejected"] += 1
            continue
        late = clock is not None and row["ts"] < clock
        if late:
            flags.append("late")                 # older than something already stored
        cur = conn.execute(
            "INSERT OR IGNORE INTO readings VALUES (?, ?, ?, ?, ?, ?, ?)",
            (row["sensor_id"], row["ts"], row["temperature_c"], row["vibration_mm_s"],
             row["belt_speed_mps"], row["drift_mm"], ",".join(flags)))
        if cur.rowcount == 0:
            summary["duplicate"] += 1            # this sensor and timestamp are already stored
            continue
        summary["stored"] += 1
        summary["late"] += 1 if late else 0
        touched.add(row["sensor_id"])

    if summary["stored"]:
        analyse(conn, touched)
    if commit:
        conn.commit()
    return summary

Writing /kaggle/working/belt-monitor/backend/ingest.py


In [17]:
!rm -f /kaggle/working/test.db*
!cd /kaggle/working/belt-monitor && BELT_DB=/kaggle/working/test.db python -c "from backend.db import connect; from backend.ingest import ingest; conn = connect(); print(ingest(conn, [dict(sensor_id='S1', ts='2026-09-01T00:00:00Z', temperature_c=46.35, vibration_mm_s=1.599, belt_speed_mps=4.012, drift_mm=6.4), dict(sensor_id='S1', ts='2026-09-01T00:01:00Z', temperature_c=999, vibration_mm_s=1.7, belt_speed_mps=4.0, drift_mm=5.0), dict(sensor_id='S99', ts='2026-09-01T00:01:00Z')])); print(ingest(conn, [dict(sensor_id='S1', ts='2026-09-01T00:00:00Z', temperature_c=46.35, vibration_mm_s=1.599, belt_speed_mps=4.012, drift_mm=6.4)])); print([tuple(r) for r in conn.execute('SELECT * FROM readings')]); print([tuple(r) for r in conn.execute('SELECT * FROM rejected')])"


{'received': 3, 'stored': 2, 'rejected': 1, 'duplicate': 0, 'late': 0}
{'received': 1, 'stored': 0, 'rejected': 0, 'duplicate': 1, 'late': 0}
[('S1', '2026-09-01T00:00:00Z', 46.35, 1.599, 4.012, 6.4, ''), ('S1', '2026-09-01T00:01:00Z', None, 1.7, 4.0, 5.0, 'out_of_range:temperature_c')]
[(1, 'unknown sensor_id', '{"sensor_id": "S99", "ts": "2026-09-01T00:01:00Z"}')]


In [18]:
!python -c "import fastapi, uvicorn; print(fastapi.__version__, uvicorn.__version__)"

0.141.1 0.54.0


In [19]:
%%writefile /kaggle/working/belt-monitor/backend/app.py
"""The HTTP layer. Each function below answers one URL.

Start it from the belt-monitor folder with:
    python -m uvicorn backend.app:app --port 8000
"""
from fastapi import Body, FastAPI

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest

app = FastAPI(title="Belt monitor")

PROCESS_ANOMALIES = ["high_temperature", "high_vibration", "belt_drift"]
SENSOR_FAULTS = ["sensor_offline", "sensor_stuck"]


@app.get("/health")
def health():
    conn = connect()
    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    clock = conn.execute("SELECT MAX(ts) FROM readings").fetchone()[0]
    conn.close()
    return {"status": "ok", "readings": count, "clock": clock}


@app.post("/readings")
def post_readings(messages: list[dict] = Body(...)):
    """The simulator sends its messages here. Each message is checked one by one,
    so one bad message cannot make the whole batch fail."""
    conn = connect()
    summary = ingest(conn, messages)
    conn.close()
    return summary


@app.get("/readings")
def get_readings(sensor_id: str, limit: int = 240):
    """The newest readings of one sensor, oldest first (for the dashboard charts)."""
    conn = connect()
    rows = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT ?",
                        (sensor_id, limit)).fetchall()
    conn.close()
    return [dict(r) for r in reversed(rows)]


@app.get("/anomalies")
def get_anomalies(active: bool = True, limit: int = 200):
    """Active anomalies, or the most recent ones of any state when active=false."""
    conn = connect()
    where = "WHERE ended_ts IS NULL" if active else ""
    rows = conn.execute(f"SELECT * FROM anomalies {where} ORDER BY started_ts DESC LIMIT ?",
                        (limit,)).fetchall()
    conn.close()
    return [dict(r) for r in rows]


@app.get("/sensors")
def get_sensors():
    """One entry per sensor point: where it is, its newest reading and its status."""
    conn = connect()
    active = conn.execute("SELECT sensor_id, type, severity FROM anomalies "
                          "WHERE ended_ts IS NULL").fetchall()
    slip = [a for a in active if a["type"] == "belt_slip"]
    result = []
    for sid in C.SENSOR_IDS:
        newest = conn.execute("SELECT * FROM readings WHERE sensor_id = ? ORDER BY ts DESC LIMIT 1",
                              (sid,)).fetchone()
        mine = [a for a in active if a["sensor_id"] == sid]
        process = [a["type"] for a in mine if a["type"] in PROCESS_ANOMALIES]
        faults = [a["type"] for a in mine if a["type"] in SENSOR_FAULTS]
        if slip:
            process.append("belt_slip")        # slip slows the belt at every sensor point
        if newest is None or "sensor_offline" in faults:
            status = "no_data"                 # no current reading: it cannot be called normal
        elif process:
            status = "abnormal"
        else:
            status = "normal"
        result.append({"sensor_id": sid, "position_m": C.SENSOR_POSITION_M[sid],
                       "status": status, "anomalies": process, "sensor_faults": faults,
                       "newest": dict(newest) if newest else None})
    conn.close()
    return result

Writing /kaggle/working/belt-monitor/backend/app.py


In [20]:
import subprocess, time
server = subprocess.Popen(["python", "-m", "uvicorn", "backend.app:app", "--port", "8000"],
                          cwd="/kaggle/working/belt-monitor",
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
!curl -s http://127.0.0.1:8000/health

{"status":"ok","readings":0,"clock":null}

In [21]:
!cd /kaggle/working/belt-monitor && python -m simulator.run live --seed 7 --days 3 --speedup 60000 --out data/live
!curl -s http://127.0.0.1:8000/health

messages sent: 42922, not delivered: 0
{"status":"ok","readings":42922,"clock":"2026-09-03T23:59:00Z"}

In [22]:
!curl -s "http://127.0.0.1:8000/anomalies?active=false&limit=2"

[{"id":12,"sensor_id":"S6","position_m":1800,"type":"sensor_stuck","severity":"warning","value":null,"started_ts":"2026-09-03T19:42:00Z","last_ts":"2026-09-03T21:21:00Z","ended_ts":"2026-09-03T21:22:00Z"},{"id":11,"sensor_id":"S6","position_m":1800,"type":"sensor_stuck","severity":"warning","value":null,"started_ts":"2026-09-03T04:09:00Z","last_ts":"2026-09-03T05:04:00Z","ended_ts":"2026-09-03T05:05:00Z"}]

In [23]:
%%writefile /kaggle/working/belt-monitor/backend/load_csv.py
"""Load a readings.csv written by the simulator into the database.

    python -m backend.load_csv data/train/readings.csv

Rows are fed to ingest() minute by minute in arrival order, so the backend
sees exactly what it would see from the live feed.
"""
import csv
import sys
import time

from simulator import config as C
from backend.db import connect
from backend.ingest import ingest


def to_message(row):
    message = {"sensor_id": row["sensor_id"], "ts": row["ts"]}
    for ch in C.CHANNELS:
        message[ch] = None if row[ch] == "" else float(row[ch])
    return message


def load(csv_path, db_path=None):
    conn = connect(db_path)
    total = {"received": 0, "stored": 0, "rejected": 0, "duplicate": 0, "late": 0}
    batch, batch_arrival, batches = [], None, 0
    started = time.time()

    def flush():
        result = ingest(conn, batch, commit=False)
        for key in total:
            total[key] += result[key]

    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["arrival_ts"] != batch_arrival and batch:
                flush()
                batch = []
                batches += 1
                if batches % 1000 == 0:
                    conn.commit()
            batch_arrival = row["arrival_ts"]
            batch.append(to_message(row))
    if batch:
        flush()
    conn.commit()
    print(f"{total} in {time.time() - started:.0f} s")
    return conn


if __name__ == "__main__":
    load(sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else None)

Writing /kaggle/working/belt-monitor/backend/load_csv.py


In [24]:
!cd /kaggle/working/belt-monitor && python -m backend.load_csv data/train/readings.csv data/train.db

{'received': 856001, 'stored': 856001, 'rejected': 0, 'duplicate': 0, 'late': 8478} in 71 s


In [25]:
import os
os.makedirs("/kaggle/working/belt-monitor/analysis", exist_ok=True)

In [26]:
%%writefile /kaggle/working/belt-monitor/analysis/backend_check.py
"""Compare what the backend stored with the simulator's ground-truth logs.

    python -m analysis.backend_check

Reads the database (filled by backend.load_csv) and data/train/*.csv.
"""
import csv
from datetime import datetime

from backend.db import connect

DATA = "data/train"
FMT = "%Y-%m-%dT%H:%M:%SZ"
conn = connect()
failures = list(csv.DictReader(open(f"{DATA}/failures.csv")))
faults = list(csv.DictReader(open(f"{DATA}/data_faults.csv")))
anomalies = [dict(r) for r in conn.execute("SELECT * FROM anomalies")]


def hours(a, b):
    return (datetime.strptime(b, FMT) - datetime.strptime(a, FMT)).total_seconds() / 3600


print("MESSY DATA")
# Impossible spikes must be stored as NULL with an out_of_range flag.
impossible = [x for x in faults if x["fault_type"] == "spike" and x["detail"].startswith("impossible")]
caught = 0
for x in impossible:
    r = conn.execute("SELECT * FROM readings WHERE sensor_id = ? AND ts = ?",
                     (x["sensor_id"], x["start_ts"])).fetchone()
    if r is not None and r[x["channel"]] is None and "out_of_range:" + x["channel"] in r["flags"]:
        caught += 1
print(f"  impossible spikes stored as NULL and flagged : {caught} of {len(impossible)}")

nulls = [x for x in faults if x["fault_type"] == "null_field"]
caught = sum(1 for x in nulls if conn.execute(
    "SELECT 1 FROM readings WHERE sensor_id = ? AND ts = ? AND flags LIKE ?",
    (x["sensor_id"], x["start_ts"], "%missing:" + x["channel"] + "%")).fetchone())
print(f"  null values flagged as missing              : {caught} of {len(nulls)}")

late = [x for x in faults if x["fault_type"] == "late"]
flagged = conn.execute("SELECT COUNT(*) FROM readings WHERE flags LIKE '%late%'").fetchone()[0]
print(f"  late readings flagged                       : {flagged} of {len(late)}")

for kind, anomaly_type in [("outage", "sensor_offline"), ("stuck", "sensor_stuck")]:
    windows = [x for x in faults if x["fault_type"] == kind]
    found = 0
    for x in windows:
        for a in anomalies:
            if (a["type"] == anomaly_type and a["sensor_id"] == x["sensor_id"]
                    and a["started_ts"] <= x["end_ts"] and (a["ended_ts"] or "9") >= x["start_ts"]):
                found += 1
                break
    extra = sum(1 for a in anomalies if a["type"] == anomaly_type and not any(
        a["sensor_id"] == x["sensor_id"] and a["started_ts"] <= x["end_ts"]
        and (a["ended_ts"] or "9") >= x["start_ts"] for x in windows))
    print(f"  {kind:6} periods with a {anomaly_type:14} anomaly: {found} of {len(windows)}"
          f"   (anomalies with no matching period: {extra})")

print("\nFAILURES   (warning = hours between the first anomaly and the failure)")
MATCH = {"seized_roller": ["high_vibration", "high_temperature"],
         "belt_drift": ["belt_drift"], "belt_slip": ["belt_slip"]}
for ftype, anomaly_types in MATCH.items():
    episodes = [f for f in failures if f["type"] == ftype]
    for anomaly_type in anomaly_types:
        leads = []
        for f in episodes:
            starts = [a["started_ts"] for a in anomalies
                      if a["type"] == anomaly_type and a["sensor_id"] == f["sensor_id"]
                      and f["onset_ts"] <= a["started_ts"] <= f["failure_ts"]]
            if starts:
                leads.append(hours(min(starts), f["failure_ts"]))
        leads.sort()
        mid = leads[len(leads) // 2] if leads else float("nan")
        # An anomaly is false when it does not overlap any episode of its failure type.
        false = sum(1 for a in anomalies if a["type"] == anomaly_type and not any(
            a["sensor_id"] == f["sensor_id"] and a["started_ts"] <= f["repaired_ts"]
            and (a["ended_ts"] or "9") >= f["onset_ts"] for f in episodes))
        total = sum(1 for a in anomalies if a["type"] == anomaly_type)
        print(f"  {ftype:14} {anomaly_type:17}: caught {len(leads)} of {len(episodes)},"
              f" median warning {mid:4.1f} h, anomalies opened {total}, false {false}")

Writing /kaggle/working/belt-monitor/analysis/backend_check.py


In [27]:
import os
os.makedirs("/kaggle/working/belt-monitor/analysis", exist_ok=True)

In [28]:
%%writefile /kaggle/working/belt-monitor/analysis/backend_check.py
"""Compare what the backend stored with the simulator's ground-truth logs.

    python -m analysis.backend_check

Reads the database (filled by backend.load_csv) and data/train/*.csv.
"""
import csv
from datetime import datetime

from backend.db import connect

DATA = "data/train"
FMT = "%Y-%m-%dT%H:%M:%SZ"
conn = connect()
failures = list(csv.DictReader(open(f"{DATA}/failures.csv")))
faults = list(csv.DictReader(open(f"{DATA}/data_faults.csv")))
anomalies = [dict(r) for r in conn.execute("SELECT * FROM anomalies")]


def hours(a, b):
    return (datetime.strptime(b, FMT) - datetime.strptime(a, FMT)).total_seconds() / 3600


print("MESSY DATA")
# Impossible spikes must be stored as NULL with an out_of_range flag.
impossible = [x for x in faults if x["fault_type"] == "spike" and x["detail"].startswith("impossible")]
caught = 0
for x in impossible:
    r = conn.execute("SELECT * FROM readings WHERE sensor_id = ? AND ts = ?",
                     (x["sensor_id"], x["start_ts"])).fetchone()
    if r is not None and r[x["channel"]] is None and "out_of_range:" + x["channel"] in r["flags"]:
        caught += 1
print(f"  impossible spikes stored as NULL and flagged : {caught} of {len(impossible)}")

nulls = [x for x in faults if x["fault_type"] == "null_field"]
caught = sum(1 for x in nulls if conn.execute(
    "SELECT 1 FROM readings WHERE sensor_id = ? AND ts = ? AND flags LIKE ?",
    (x["sensor_id"], x["start_ts"], "%missing:" + x["channel"] + "%")).fetchone())
print(f"  null values flagged as missing              : {caught} of {len(nulls)}")

late = [x for x in faults if x["fault_type"] == "late"]
flagged = conn.execute("SELECT COUNT(*) FROM readings WHERE flags LIKE '%late%'").fetchone()[0]
print(f"  late readings flagged                       : {flagged} of {len(late)}")

for kind, anomaly_type in [("outage", "sensor_offline"), ("stuck", "sensor_stuck")]:
    windows = [x for x in faults if x["fault_type"] == kind]
    found = 0
    for x in windows:
        for a in anomalies:
            if (a["type"] == anomaly_type and a["sensor_id"] == x["sensor_id"]
                    and a["started_ts"] <= x["end_ts"] and (a["ended_ts"] or "9") >= x["start_ts"]):
                found += 1
                break
    extra = sum(1 for a in anomalies if a["type"] == anomaly_type and not any(
        a["sensor_id"] == x["sensor_id"] and a["started_ts"] <= x["end_ts"]
        and (a["ended_ts"] or "9") >= x["start_ts"] for x in windows))
    print(f"  {kind:6} periods with a {anomaly_type:14} anomaly: {found} of {len(windows)}"
          f"   (anomalies with no matching period: {extra})")

print("\nFAILURES   (warning = hours between the first anomaly and the failure)")
MATCH = {"seized_roller": ["high_vibration", "high_temperature"],
         "belt_drift": ["belt_drift"], "belt_slip": ["belt_slip"]}
for ftype, anomaly_types in MATCH.items():
    episodes = [f for f in failures if f["type"] == ftype]
    for anomaly_type in anomaly_types:
        leads = []
        for f in episodes:
            starts = [a["started_ts"] for a in anomalies
                      if a["type"] == anomaly_type and a["sensor_id"] == f["sensor_id"]
                      and f["onset_ts"] <= a["started_ts"] <= f["failure_ts"]]
            if starts:
                leads.append(hours(min(starts), f["failure_ts"]))
        leads.sort()
        mid = leads[len(leads) // 2] if leads else float("nan")
        # An anomaly is false when it does not overlap any episode of its failure type.
        false = sum(1 for a in anomalies if a["type"] == anomaly_type and not any(
            a["sensor_id"] == f["sensor_id"] and a["started_ts"] <= f["repaired_ts"]
            and (a["ended_ts"] or "9") >= f["onset_ts"] for f in episodes))
        total = sum(1 for a in anomalies if a["type"] == anomaly_type)
        print(f"  {ftype:14} {anomaly_type:17}: caught {len(leads)} of {len(episodes)},"
              f" median warning {mid:4.1f} h, anomalies opened {total}, false {false}")

Overwriting /kaggle/working/belt-monitor/analysis/backend_check.py


In [29]:
!cd /kaggle/working/belt-monitor && BELT_DB=/kaggle/working/belt-monitor/data/train.db python -m analysis.backend_check

MESSY DATA
  impossible spikes stored as NULL and flagged : 845 of 845
  null values flagged as missing              : 2516 of 2516
  late readings flagged                       : 8478 of 8478
  outage periods with a sensor_offline anomaly: 53 of 53   (anomalies with no matching period: 0)
  stuck  periods with a sensor_stuck   anomaly: 60 of 60   (anomalies with no matching period: 0)

FAILURES   (warning = hours between the first anomaly and the failure)
  seized_roller  high_vibration   : caught 27 of 27, median warning  8.3 h, anomalies opened 56, false 0
  seized_roller  high_temperature : caught 27 of 27, median warning  5.1 h, anomalies opened 32, false 0
  belt_drift     belt_drift       : caught 29 of 29, median warning  3.1 h, anomalies opened 42, false 0
  belt_slip      belt_slip        : caught 30 of 30, median warning  4.1 h, anomalies opened 30, false 0
